<a href="https://colab.research.google.com/github/tjdux/basic_of_ml/blob/main/29_%EC%9D%B4%EB%AF%B8%EC%A7%80_%EB%B6%84%EB%A5%98_%EB%AA%A8%EB%8D%B8%EC%9D%98_%ED%9A%A8%EC%9C%A8%EC%84%B1_%EC%B5%9C%EC%A0%81%ED%99%94%ED%95%98%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ResNet의 확장 모델 - DenseNet
- ResNet: 스킵 연결을 사용해 기존 층을 건너뛰어 마지막 합성곱 출력에 입력을 더함
- DenseNet: 이전 층의 모든 출력을 현재 층의 입력으로 사용 (**밀집 연결** dense connectivity)
- 연결층을 사용하여 차곡차곡 입력을 쌓아 다음 밀집 블록으로 계속 전달 👉 그레이디언트 소실 문제 해결

### DenseNet 모델 준비하기


In [13]:
# 밀집 블록
import keras
from keras import layers

def dense_block(x, blocks):
  for _ in range(blocks):
    x1 = layers.BatchNormalization(epsilon=1e-5)(x)
    x1 = layers.Activation('relu')(x1)

    x1 = layers.Conv2D(filters=128, kernel_size=1, use_bias=False)(x1)

    x1 = layers.BatchNormalization(epsilon=1e-5)(x1)
    x1 = layers.Activation('relu')(x1)

    x1 = layers.Conv2D(filters=32, kernel_size=3, padding="same",
                       use_bias=False)(x1)

    x = layers.Concatenate()([x, x1]) # 연결층
  return x

- **전환 블록**(transition block): 특성 맵의 너비와 높이, 채널을 줄여줌

In [14]:
# 전환 블록
def transition_block(x):
  x = layers.BatchNormalization(epsilon=1e-5)(x)
  x = layers.Activation('relu')(x)

  x = layers.Conv2D(int(x.shape[-1]/2), 1, use_bias=False)(x)

  x = layers.AveragePooling2D(2)(x)

  return x

### DenseNet 모델 만들기 (DenseNet-121)

In [15]:
inputs = layers.Input(shape=(224, 224, 3))

In [16]:
x = layers.ZeroPadding2D(padding=3)(inputs)
x = layers.Conv2D(filters=64, kernel_size=7, strides=2, use_bias=False)(x)

In [17]:
x = layers.BatchNormalization(epsilon=1e-5)(x)
x = layers.Activation('relu')(x)
x = layers.ZeroPadding2D(padding=1)(x)
x = layers.MaxPooling2D(3, strides=2)(x)

In [18]:
for blocks in (6, 12, 24):
  x = dense_block(x, blocks)
  x = transition_block(x)

x = dense_block(x, 16)

In [20]:
x = layers.BatchNormalization(epsilon=1e-5)(x)
x = layers.Activation('relu')(x)
x = layers.GlobalAveragePooling2D()(x)
outputs = layers.Dense(1000, activation='softmax')(x)

In [21]:
model = keras.Model(inputs, outputs)

In [22]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d_4    │ (None, 230, 230,  │          0 │ input_layer_2[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 112, 112,  │      9,408 │ zero_padding2d_4… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 112, 112,  │        256 │ conv2d_6[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_6        │ (None, 112, 112,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d_5    │ (None, 114, 114,  │          0 │ activation_6[0][… │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 56, 56,    │          0 │ zero_padding2d_5… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 56, 56,    │        256 │ max_pooling2d_2[… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_7        │ (None, 56, 56,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 56, 56,    │      8,192 │ activation_7[0][… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 56, 56,    │        512 │ conv2d_7[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_8        │ (None, 56, 56,    │          0 │ batch_normalizat… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_8 (Conv2D)   │ (None, 56, 56,    │     36,864 │ activation_8[0][… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, 56, 56,    │          0 │ max_pooling2d_2[… │
│ (Concatenate)       │ 96)               │            │ conv2d_8[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 56, 56,    │        384 │ concatenate_2[0]… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_9        │ (None, 56, 56,    │          0 │ batch_normalizat… │
│ (Activation)        │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_9 (Conv2D)   │ (None, 56, 56,    │     12,288 │ activation_9[0][

 Total params: 8,066,600 (30.77 MB)

 Trainable params: 7,980,904 (30.44 MB)

 Non-trainable params: 85,696 (334.75 KB)

### DenseNet 모델로 강아지 사진 분류하기

In [23]:
!gdown 1xGkTT3uwYt4myj6eJJeYtdEFgTi2Sj8C
!unzip cat-dog-images.zip

Downloading...
From: https://drive.google.com/uc?id=1xGkTT3uwYt4myj6eJJeYtdEFgTi2Sj8C
To: /content/cat-dog-images.zip
100% 182k/182k [00:00<00:00, 104MB/s]
Archive:  cat-dog-images.zip
   creating: images/
  inflating: images/dog.png          
  inflating: images/cat.png          


In [24]:
import numpy as np
from PIL import Image
from keras.applications import densenet

dog_png = np.array(Image.open('images/dog.png'))

dense_prep_dog = densenet.preprocess_input(dog_png)

In [25]:
densenet201 = keras.applications.DenseNet201()
predictions = densenet201.predict(dense_prep_dog[np.newaxis, :])
densenet.decode_predictions(predictions)

82524592/82524592 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 28s 28s/step
35363/35363 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step


[[('n02099712', 'Labrador_retriever', np.float32(0.52812004)),
  ('n04409515', 'tennis_ball', np.float32(0.19857836)),
  ('n02104029', 'kuvasz', np.float32(0.060605414)),
  ('n02111500', 'Great_Pyrenees', np.float32(0.02718823)),
  ('n02099601', 'golden_retriever', np.float32(0.017613882))]]

## 모바일 환경(경량) 모델 - MobileNet
- **깊이별 합성곱층**(depthwise convolution) 이라는 방식으로 용량 문제 해결
- 각 채널마다 합성곱이 따로 따로 수행되는 필터를 사용하여 각 채널의 특성을 독립적으로 추출
- 별도의 필터와 단순한 연산, 각 채널의 독립적 처리를 통해 메모리와 처리 성능이 제한적인 환경에서 높은 성능
- 깊이별 합성곱: 입력 채널의 개수 == 출력 채널의 개수 (필터 개수 == 입력 채널의 개수)


### MobileNet 모델 준비하기
- **깊이별 분리 합성곱 블록**(depthwise separable convolution block)을 반복적으로 쌓아서 구성

In [27]:
# 깊이별 분리 합성곱 블록
def depthwise_separable_block(inputs, filters, strides=1):
  if strides == 1:
    x = inputs
  else:
    x = layers.ZeroPadding2D(padding=((0, 1), (1, 0)))(inputs)
    # 위에 0픽셀, 아래에 1픽셀, 왼쪽에 0픽셀, 오른쪽에 1픽셀

  x = layers.DepthwiseConv2D(kernel_size=3, padding='same' if strides == 1 else 'valid',
                            strides=strides, use_bias=False)(x)
  x = layers.BatchNormalization(epsilon=1e-5)(x)
  x = layers.ReLU(max_value=6.0)(x)
  x = layers.Conv2D(filters, 1, padding='same', use_bias=False)(x)
  x = layers.BatchNormalization(epsilon=1e-5)(x)
  x = layers.ReLU(max_value=6.0)(x)

  return x

### MobileNet 모델 만들기

In [28]:
inputs = layers.Input(shape=(224, 224, 3))

In [29]:
x = layers.Conv2D(32, 3, padding='same', strides=2, use_bias=False)(inputs)
x = layers.BatchNormalization(epsilon=1e-5)(x)
x = layers.ReLU(max_value=6.0)(x)

In [30]:
for filters in (64, 128, 256):
  x = depthwise_separable_block(x, filters)
  x = depthwise_separable_block(x, filters*2, strides=2)

In [31]:
for _ in range(5):
  x = depthwise_separable_block(x, 512)
x = depthwise_separable_block(x, 1024, strides=2)
x = depthwise_separable_block(x, 102)

In [33]:
x = layers.GlobalAveragePooling2D(keepdims=True)(x) #(1, 1, 1024)

In [34]:
x = layers.Dropout(0.001)(x)
x = layers.Conv2D(1000, 1, padding='same')(x)
x = layers.Reshape((1000,))(x)
outputs = layers.Activation('softmax')(x)

In [35]:
model = keras.Model(inputs, outputs)

In [36]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_126 (Conv2D)             │ (None, 112, 112, 32)   │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_128         │ (None, 112, 112, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d                │ (None, 112, 112, 32)   │           288 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_129         │ (None, 112, 112, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_127 (Conv2D)             │ (None, 112, 112, 64)   │         2,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_130         │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ zero_padding2d_8                │ (None, 113, 113, 64)   │             0 │
│ (ZeroPadding2D)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_1              │ (None, 56, 56, 64)     │           576 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_131         │ (None, 56, 56, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_3 (ReLU)                  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_128 (Conv2D)             │ (None, 56, 56, 128)    │         8,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_132         │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_4 (ReLU)                  │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_2              │ (None, 56, 56, 128)    │         1,152 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_133         │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_5 (ReLU)                  │ (None, 56, 56, 128)    │             

 Total params: 2,384,048 (9.09 MB)

 Trainable params: 2,364,004 (9.02 MB)

 Non-trainable params: 20,044 (78.30 KB)

### MobileNet 모델로 강아지 사진 분류하기

In [37]:
from keras.applications import mobilenet

mobile_prep_dog = mobilenet.preprocess_input(dog_png)
model = keras.applications.MobileNet()
predictions = model.predict(mobile_prep_dog[np.newaxis, :])
mobilenet.decode_predictions(predictions)

17225924/17225924 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 11s 11s/step


[[('n02099712', 'Labrador_retriever', np.float32(0.40903902)),
  ('n02104029', 'kuvasz', np.float32(0.18954752)),
  ('n02110341', 'dalmatian', np.float32(0.14881587)),
  ('n02111500', 'Great_Pyrenees', np.float32(0.04276249)),
  ('n02099601', 'golden_retriever', np.float32(0.027608197))]]